In [1]:
#conda activate burnseverity
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

import requests
import json

import geopandas as gpd

import rasterio as rio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.mask import mask
from rasterio import features
from rasterio.plot import show_hist

from shapely.geometry import shape, mapping
from shapely.ops import unary_union

import validation as val


SyntaxError: unterminated string literal (detected at line 194) (validation.py, line 194)

### Validation goals

#### Questions
1. Difference between our dNBR and dNBR from BAIR?
2. Are the two different indices truly different in our tool and is ours truly more sensitive?
3. Do our boundaries and those from CalFire match?
4. Sensitivity around chosen time windows?

#### Metrics
1. Recoarse Sentinel to Landsat, $R^2$
2. $var(dNBR) < var(RBR)$
3. Percent overlap
4. Before period fixed through three weeks. After ${5, 10, 15, 21, 30, 45, 60, 90}$. Criteria: stable and low variance 

In [2]:
fires = pd.read_csv('fire_processing_jobs.csv')
calfire = gpd.read_file('Validation_Fire_Perimeters_2015_2024.shp')

In [ ]:
# Process all fires and date ranges
results_list = []
current_fire = None

for _, row in fires.iterrows():
    # Print only when starting a new fire
    if row['fire_name'] != current_fire:
        print(f"Processing {row['fire_name']}...")
        current_fire = row['fire_name']
    
    result = val.process_fire_metrics(row['fire_name'], row['post_fire_days'], fires, calfire)
    
    if result is False:
        continue
    
    # result is now a list of dicts (one per metric)
    results_list.extend(result)

# Create DataFrame
metrics_df = pd.DataFrame(results_list)

# Split by metric and save separate geopackages
dnbr_df = metrics_df[metrics_df['metric'] == 'dnbr'].copy()
rdnbr_df = metrics_df[metrics_df['metric'] == 'rdnbr'].copy()

# Save dNBR as GeoPackage
dnbr_gdf = gpd.GeoDataFrame(
    dnbr_df,
    geometry='filtered_polygon',
    crs='EPSG:4326'
)
dnbr_gdf.to_file('validation_metrics_dnbr.gpkg', driver='GPKG')

# Save RdNBR as GeoPackage
rdnbr_gdf = gpd.GeoDataFrame(
    rdnbr_df,
    geometry='filtered_polygon',
    crs='EPSG:4326'
)
rdnbr_gdf.to_file('validation_metrics_rdnbr.gpkg', driver='GPKG')

# Also save combined attributes as CSV for quick viewing
metrics_df.drop('filtered_polygon', axis=1).to_csv('validation_metrics.csv', index=False)

print(f"\nFinal table shape: {metrics_df.shape}")
print(f"dNBR rows: {len(dnbr_df)}, RdNBR rows: {len(rdnbr_df)}")
print("Saved spatial data to validation_metrics_dnbr.gpkg and validation_metrics_rdnbr.gpkg")
print("Saved CSV to validation_metrics.csv")

metrics_df.head()

Processing COFFEE POT...
Processing SENTINEL...
Processing SIMPSON...
Processing YORK...
  Job YORK_2023-07-28_5 is pending
  Job YORK_2023-07-28_10 is pending
  Job YORK_2023-07-28_15 is pending
  Job YORK_2023-07-28_21 is pending
  Job YORK_2023-07-28_30 is pending
  Job YORK_2023-07-28_45 is pending
  Job YORK_2023-07-28_60 is pending
  Job YORK_2023-07-28_90 is pending
Processing REDWOOD...


/opt/miniconda3/envs/burnseverity/lib/python3.13/site-packages/numpy/_core/_methods.py:188: RuntimeWarning: invalid value encountered in subtract
  x = um.subtract(arr, arrmean, out=...)
/opt/miniconda3/envs/burnseverity/lib/python3.13/site-packages/numpy/_core/_methods.py:188: RuntimeWarning: invalid value encountered in subtract
  x = um.subtract(arr, arrmean, out=...)


Processing GEOLOGY...
Processing VALLEY...
Processing SYCAMORE...
Processing SUMMIT...
Processing ELK TRAIL...
Processing AVALANCHE...
Processing KNP Complex...
  Job KNP Complex_2021-09-10_5 is pending
  Job KNP Complex_2021-09-10_10 is pending
  Job KNP Complex_2021-09-10_15 is pending
  Job KNP Complex_2021-09-10_21 is pending
  Job KNP Complex_2021-09-10_30 is pending
  Job nan is pending
  Job nan is pending
  Job nan is pending
Processing MOJAVE...
  Job nan is pending
Processing POND...
Processing LOST...
Processing HART...
Processing CASTLE...
Processing DOME...
  Job DOME_2020-08-15_5 is pending
  Job DOME_2020-08-15_10 is pending
  Job DOME_2020-08-15_15 is pending
  Job DOME_2020-08-15_21 is pending
  Job DOME_2020-08-15_30 is pending
  Job DOME_2020-08-15_45 is pending
  Job DOME_2020-08-15_60 is pending
  Job DOME_2020-08-15_90 is pending
Processing RATTLESNAKE...
Processing SCORPION...
Processing MORAINE...
Processing IVANPAH...
Processing BULL...
Processing WENDY...
Proc

,fire_name,fire_days,fire_event_name,metric,calfire_mean,calfire_var,filtered_mean,filtered_var,filtered_polygon
0,COFFEE POT,5,COFFEE POT_2024-08-03_5,dnbr,0.058276,0.008079,0.076561,0.007665,MULTIPOLYGON (((-118.78054641229755 36.3526655...
1,COFFEE POT,5,COFFEE POT_2024-08-03_5,rdnbr,0.064776,0.007889,0.086878,0.005868,MULTIPOLYGON (((-118.78054641229755 36.3526655...
2,COFFEE POT,10,COFFEE POT_2024-08-03_10,dnbr,0.055370,0.007539,0.074775,0.006698,MULTIPOLYGON (((-118.7948966008226 36.35266556...
3,COFFEE POT,10,COFFEE POT_2024-08-03_10,rdnbr,0.065616,0.008669,0.089038,0.006691,MULTIPOLYGON (((-118.7948966008226 36.35266556...
4,COFFEE POT,15,COFFEE POT_2024-08-03_15,dnbr,0.028381,0.009027,0.064690,0.006796,MULTIPOLYGON (((-118.79507597817916 36.3524830...
